## O mesmo trabalho, dois modelos

Toda chamada deste repositório passa pelo `llm.chat()`, e agora ele aceita
`model=`. Isso abre a pergunta que fecha a demo-3: **por que o modelo caro
responderia "diga ok"?**

A proposta do **model-route** é simples: a mesma resposta, com uma conta
menor. Em vez de escolher o modelo por aplicação (ou por hábito), o código
pergunta a cada mensagem quem deve responder. O juiz System One **rotula** a
mensagem — um `Choice` entre `mini` e `forte`, com probabilidades e
confiança — e a **política de confiança no código** decide o que fazer com
esse rótulo. O modelo mede; o código decide.

O roteiro tem 6 prompts, 3 leves e 3 moderados, e o notebook roda duas
vezes:

1. **all-strong** — as 6 perguntas no modelo forte, sem juiz. É o baseline.
2. **routed** — as mesmas 6, cada uma com uma chamada ao Jev decidindo o
   modelo. O custo da decisão entra na conta.

No fim, um juiz de qualidade (`Noul`) compara as duas respostas de cada
pergunta: economia só vale se a resposta roteada concordar com a forte.

O modelo continua peça substituível: `LLM_MODEL` e `LLM_MINI_MODEL` vêm do
`.env`. Trocar de provedor é mexer no arquivo, não no notebook.


In [1]:
# Setup: os dois modelos vêm do .env e o roteiro é fixo - os mesmos 6
# prompts em todas as rodadas, senão a comparação de custo não vale.
import os, pathlib, sys
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import jev
import llm
from llm import chat, last_usage

# Fallbacks comentados: o .env sempre ganha. deepseek-v4.1-flash é o
# "forte" do .env de referência; glm-5.3-flash é o default do .env.example
# (os ids -free testados responderam 400 no path do plano Go).
FORTE = os.environ.get("LLM_MODEL") or "deepseek-v4.1-flash"
MINI = os.environ.get("LLM_MINI_MODEL") or "glm-5.3-flash"

print(f"Forte: {FORTE}")
print(f"Mini:  {MINI}")
print(f"Juiz:  {os.environ.get('JEV_MODEL') or jev._DEFAULT_MODEL}")

df = pd.read_csv('../data/reclamacoes.csv')
RECLAMACOES = "\n".join(f"- {texto}" for texto in df['texto'])

ROTEIRO = [
    {"tipo": "leve", "pergunta": "Diga apenas: ok."},
    {"tipo": "leve", "pergunta": "Escreva um haiku sobre prazos de entrega."},
    {"tipo": "leve", "pergunta": "Formate esta frase em minúsculas: O PRAZO É O QUE MAIS IMPORTA."},
    {"tipo": "moderada", "pergunta": "Depois da última atualização o app nem abre: qual categoria classifica melhor esta reclamação?"},
    {"tipo": "moderada", "pergunta": "Compare esta resposta com o gabarito e diga o que falta. Resposta: 'o pedido atrasou'. Gabarito: 'o pedido atrasou 3 dias porque o centro de distribuição estava sem estoque'."},
    {"tipo": "moderada", "pergunta": "Resuma as 18 reclamações em 3 bullets:\n" + RECLAMACOES},
]
print(f"\n{len(ROTEIRO)} prompts: {sum(i['tipo'] == 'leve' for i in ROTEIRO)} leves, {sum(i['tipo'] == 'moderada' for i in ROTEIRO)} moderados")


Forte: deepseek-v4.1-flash
Mini:  glm-5.3-flash
Juiz:  jev-1.13-free

6 prompts: 3 leves, 3 moderados


In [2]:
# A pergunta de rota é um Choice: o Jev devolve o critério escolhido, a
# confiança e as probabilidades. Os critérios são a política do roteador -
# texto nosso, calibrável sem trocar de modelo.
ROTA = jev.Choice(
    "Qual modelo deve responder esta mensagem?",
    {
        "mini": "Saudação, formatação ou tarefa curta e óbvia: baixo risco de julgamento sutil.",
        "forte": "Múltiplos fatores, sentimento ou precisão contextual: um erro aqui custa caro.",
    },
)

# Regra no código, não no modelo: confiança abaixo de 0.7 escala para o
# forte. É o freio barato do roteador - o Jev mede, o código decide.
LIMIAR_CONFIANCA = 0.7


def delta(antes, depois):
    """Diferença entre dois retratos de usage_total() (chat ou Jev)."""
    return {k: depois[k] - antes[k] for k in antes}


def decidir(pergunta, rota=ROTA):
    """Uma chamada ao Jev por mensagem; devolve (modelo, decisão)."""
    resposta = jev.system_one(pergunta, {"rota": rota})["rota"]
    confianca = resposta.get("confianca")
    final = resposta["valor"]
    ajustada = False

    # Sem confiança declarada também é motivo para escalar: na dúvida,
    # paga-se o forte - a mesma lógica do threshold de revisão do 01.
    if confianca is None or confianca < LIMIAR_CONFIANCA:
        final = "forte"
        ajustada = True

    return (FORTE if final == "forte" else MINI), {
        "rota": resposta["valor"],
        "final": final,
        "confianca": confianca,
        "probabilidades": resposta.get("probabilidades"),
        "ajustada": ajustada,
    }


def rodada(itens, seletor=None):
    """Roda o roteiro inteiro; sem seletor, tudo no forte (baseline)."""
    respostas, por_modelo = [], {}

    for i, item in enumerate(itens, 1):
        if seletor is None:
            modelo, decisao = FORTE, None
        else:
            modelo, decisao = seletor(item["pergunta"])

        # 6000 e não 1200: os flashes alocam parte do orçamento pensando
        # antes de escrever; com 1600 uma execução truncou e com 3000 outra.
        texto = chat(
            [{"role": "user", "content": item["pergunta"]}],
            temperature=0,
            max_tokens=6000,
            model=modelo,
        )
        uso = last_usage() or {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
        acumulado = por_modelo.setdefault(modelo, {"prompt_tokens": 0, "completion_tokens": 0})
        acumulado["prompt_tokens"] += uso["prompt_tokens"]
        acumulado["completion_tokens"] += uso["completion_tokens"]

        respostas.append({**item, "modelo": modelo, "decisao": decisao, "resposta": texto, "uso": uso})

        if decisao is None:
            print(f"[{i}/{len(itens)}] {item['tipo']:<8} | {modelo} | {uso['total_tokens']} tok")
        else:
            conf = "?" if decisao["confianca"] is None else f"{decisao['confianca']:.2f}"
            probs = ", ".join(f"{k}={v:.2f}" for k, v in (decisao["probabilidades"] or {}).items())
            aviso = " [regra: confiança baixa]" if decisao["ajustada"] else ""
            print(f"[{i}/{len(itens)}] {item['tipo']:<8} | rota={decisao['rota']} conf={conf} ({probs}) -> {modelo}{aviso} | {uso['total_tokens']} tok")
        print(f"          {' '.join(texto.split())[:110]}")

    return {"respostas": respostas, "por_modelo": por_modelo}


In [3]:
# Rodada 1 - baseline: as 6 no forte, sem juiz. É o custo que o
# roteamento precisa bater (e o teto de qualidade que ele precisa manter).
# O baseline fecha a conta pelo usage_total(): o mesmo contador global que
# os outros notebooks usam, do início ao fim da rodada.
LLM_FORTE_ANTES = llm.usage_total()
R_FORTE = rodada(ROTEIRO)
LLM_FORTE = delta(LLM_FORTE_ANTES, llm.usage_total())
print()
for modelo, t in R_FORTE["por_modelo"].items():
    print(f"{modelo}: {t['prompt_tokens']} in / {t['completion_tokens']} out")
print(f"usage_total da rodada: {LLM_FORTE['prompt_tokens']} in / {LLM_FORTE['completion_tokens']} out")


[1/6] leve     | deepseek-v4.1-flash | 78 tok
          ok.


[2/6] leve     | deepseek-v4.1-flash | 1441 tok
          Chega o prazo O relógio não para Falta entregar


[3/6] leve     | deepseek-v4.1-flash | 112 tok
          o prazo é o que mais importa.


[4/6] moderada | deepseek-v4.1-flash | 308 tok
          A melhor classificação é: **Categoria:** Problemas técnicos / Falha no aplicativo **Subcategoria:** Aplicativo


[5/6] moderada | deepseek-v4.1-flash | 261 tok
          A resposta **“o pedido atrasou”** está correta, mas está **incompleta**. Falta acrescentar: - **Quantos dias a


[6/6] moderada | deepseek-v4.1-flash | 1583 tok
          - **Entrega e produto:** atrasos e rastreamento parado; prazo de 3 dias úteis não cumprido; pedido devolvido a

deepseek-v4.1-flash: 699 in / 3084 out
usage_total da rodada: 699 in / 3084 out


In [4]:
# Rodada 2 - routed: uma chamada ao Jev por mensagem decide o modelo.
# O custo do gate entra na conta: rotear não é de graça.
LLM_ROTA_ANTES = llm.usage_total()
JEV_ROTA_ANTES = jev.usage_total()
R_ROTA = rodada(ROTEIRO, seletor=decidir)
LLM_ROTA = delta(LLM_ROTA_ANTES, llm.usage_total())
JEV_ROTA = delta(JEV_ROTA_ANTES, jev.usage_total())

print()
for modelo, t in R_ROTA["por_modelo"].items():
    print(f"{modelo}: {t['prompt_tokens']} in / {t['completion_tokens']} out")
print(f"usage_total do chat: {LLM_ROTA['prompt_tokens']} in / {LLM_ROTA['completion_tokens']} out")
print(f"gates: {JEV_ROTA['input_tokens']} in / {JEV_ROTA['output_tokens']} out | custo US$ {JEV_ROTA['cost']:.6f}")
print(f"mini respondeu {sum(r['modelo'] == MINI for r in R_ROTA['respostas'])}/6; forte, {sum(r['modelo'] == FORTE for r in R_ROTA['respostas'])}/6")


[1/6] leve     | rota=mini conf=1.00 (mini=1.00, forte=0.00) -> glm-5.3-flash | 81 tok
          ok.


[2/6] leve     | rota=mini conf=1.00 (mini=1.00, forte=0.00) -> glm-5.3-flash | 50 tok
          Prazo apertado, o relógio não perdoa — café e coragem.


[3/6] leve     | rota=mini conf=1.00 (mini=1.00, forte=0.00) -> glm-5.3-flash | 44 tok
          o prazo é o que mais importa.


[4/6] moderada | rota=mini conf=0.64 (mini=0.82, forte=0.18) -> deepseek-v4.1-flash [regra: confiança baixa] | 414 tok
          A melhor categoria é: **“Problema técnico / falha no aplicativo”**, especificamente **“app não abre após a atu


[5/6] moderada | rota=mini conf=0.72 (mini=0.86, forte=0.14) -> glm-5.3-flash | 321 tok
          # Comparação **Resposta:** "o pedido atrasou" **Gabarito:** "o pedido atrasou 3 dias porque o centro de distri


[6/6] moderada | rota=mini conf=0.72 (mini=0.86, forte=0.14) -> glm-5.3-flash | 594 tok
          # Resumo das 18 reclamações - **Problemas de entrega e rastreamento:** rastreador parado há semanas, prazos nã

glm-5.3-flash: 536 in / 554 out
deepseek-v4.1-flash: 51 in / 363 out
usage_total do chat: 587 in / 917 out
gates: 2558 in / 192 out | custo US$ 0.000000
mini respondeu 5/6; forte, 1/6


## O Choice rotulou; a regra de confiança decidiu

O output acima é o recado mais importante do notebook: nesta execução o
`Choice` escolheu `mini` nas 6 mensagens, e a única escalada para o forte
veio da regra numérica (`confianca < 0.7`), não do rótulo tipado. Neste
sample, portanto, o roteador ativo foi a política de
confiança; a metade "tipada" do juiz foi a mais fraca das duas.

Isso não invalida o desenho — o `Choice` entrega as probabilidades que a
regra usa —, mas mostra onde calibrar: os critérios do `mini` estão largos
demais para separar tarefa trivial de julgamento sutil. O próximo passo é
reescrever esses critérios e medir de novo (o "Experimente" faz o inverso:
afrouxa ainda mais para ver o efeito no custo e na concordância).

O juiz de qualidade abaixo lê até 6000 caracteres de cada resposta — o
suficiente para cobrir a maior resposta desta execução (~4000 caracteres no
resumo do forte), sem julgar em evidência truncada.


In [5]:
# Juiz de qualidade: por pergunta, um Noul compara as duas respostas.
# "Concordar no essencial" é a régua mínima do roteamento - o mini pode
# escrever diferente, não pode entregar outra coisa.
JUIZ = jev.Noul("As duas respostas (a do mini e a do forte) concordam no essencial?")
LIMIAR_CONCORDA = 0.5

# 6000 caracteres cobrem a maior resposta desta execução (~4000 no resumo
# do forte): o juiz não deve julgar em evidência truncada.
LIMITE_JULGAMENTO = 6000


def pct(valor):
    """Porcentagem legível; "n/a" quando não há caso no denominador."""
    return "n/a" if valor != valor else f"{valor:.0%}"


def concordancia(base, roteada):
    """Julga os 6 pares (forte x roteada) e devolve taxas e custo do juiz."""
    antes = jev.usage_total()
    pares = []

    for a, b in zip(base["respostas"], roteada["respostas"]):
        estado = (
            f"Pergunta: {a['pergunta']}\n\n"
            f"Resposta A:\n{a['resposta'][:LIMITE_JULGAMENTO]}\n\n"
            f"Resposta B:\n{b['resposta'][:LIMITE_JULGAMENTO]}"
        )
        nota = jev.system_one(estado, {"concordam": JUIZ})["concordam"]["valor"]
        rota = b["decisao"]["rota"] if b["decisao"] else "forte"
        escalado = bool(b["decisao"] and b["decisao"]["ajustada"])
        pares.append({"tipo": a["tipo"], "rota": rota, "modelo": b["modelo"], "escalado": escalado, "nota": nota, "concorda": nota >= LIMIAR_CONCORDA})

        # Par forte x forte (escalado ou roteado ao forte) não mede o mini:
        # fica marcado no print e fora da manchete.
        marca = ""
        if b["modelo"] != MINI:
            motivo = "escalado pela confiança" if escalado else "rota forte"
            marca = f" ({motivo}: par forte x forte)"
        print(f"{'ok  ' if pares[-1]['concorda'] else 'NAO '} {a['tipo']:<8} rota={rota:<5} -> {b['modelo']:<20}{marca} nota={nota:.2f}")

    uso = delta(antes, jev.usage_total())
    # A taxa do mini usa quem RESPONDEU, não a escolha do Jev: um passo
    # roteado ao mini e escalado pela confiança conta como forte.
    mini = [p for p in pares if p["modelo"] == MINI]
    return {
        "pares": pares,
        "taxa_mini": sum(p["concorda"] for p in mini) / len(mini) if mini else float("nan"),
        "taxa_geral": sum(p["concorda"] for p in pares) / len(pares),
        "uso": uso,
    }


CONCORDA = concordancia(R_FORTE, R_ROTA)
mini = [p for p in CONCORDA["pares"] if p["modelo"] == MINI]
fortes = [p for p in CONCORDA["pares"] if p["modelo"] != MINI]
print(f"\nConcordância nos passos do mini: {pct(CONCORDA['taxa_mini'])} "
      f"({sum(p['concorda'] for p in mini)}/{len(mini)})")
print(f"Geral (linha secundária, inclui {len(fortes)} par(es) forte x forte): "
      f"{pct(CONCORDA['taxa_geral'])} ({sum(p['concorda'] for p in CONCORDA['pares'])}/{len(CONCORDA['pares'])})")


ok   leve     rota=mini  -> glm-5.3-flash        nota=0.99


ok   leve     rota=mini  -> glm-5.3-flash        nota=0.82


ok   leve     rota=mini  -> glm-5.3-flash        nota=0.99


ok   moderada rota=mini  -> deepseek-v4.1-flash  (escalado pela confiança: par forte x forte) nota=0.98


ok   moderada rota=mini  -> glm-5.3-flash        nota=0.97


ok   moderada rota=mini  -> glm-5.3-flash        nota=0.95

Concordância nos passos do mini: 100% (5/5)
Geral (linha secundária, inclui 1 par(es) forte x forte): 100% (6/6)


## Tokens não são custo

O roteamento corta a maior parte dos tokens de **chat**, mas acrescenta os
tokens do **gate** — por isso a célula abaixo imprime as duas contas
separadas: "Tokens (só chat)" e "Tokens (com gates)". Nesta execução o total
com gates ficou **acima** do baseline; ainda assim o custo caiu, porque o
`jev-1.13-free` custa zero.

Se o gate fosse o `jev-1.13` pago (US$ 0,042 por milhão de tokens de entrada,
output grátis), o custo do gate deixaria de ser zero e a economia cairia —
mas continuaria valendo. A conta exata está no código e é impressa na linha
"Cenário gate pago" abaixo.

In [6]:
# Preços publicados no console do OpenCode (tabela "Pricing" em
# https://opencode.ai/docs/zen, consultada em 2026-10-28), em US$ por 1M
# tokens: (input, output). O usage do chat não traz cost - a conta é
# tokens reais x preço publicado. É ESTIMATIVA, não fatura.
PRECOS = {
    "deepseek-v4.1-flash": (0.30, 1.20),
    "glm-5.3-flash": (0.15, 0.50),
}
PRECO_JEV_PAGO = 0.042  # US$/1M de input no jev-1.13; output grátis (console)


def custo_chat(por_modelo):
    """Fórmula: sum(tokens/1e6 * preço) por modelo."""
    total = 0.0
    for modelo, t in por_modelo.items():
        if modelo not in PRECOS:
            print(f"(sem preço tabelado para {modelo}; fora da conta)")
            continue
        p_in, p_out = PRECOS[modelo]
        total += t["prompt_tokens"] / 1e6 * p_in + t["completion_tokens"] / 1e6 * p_out
    return total


def custo_respostas(respostas):
    por_modelo = {}
    for r in respostas:
        acc = por_modelo.setdefault(r["modelo"], {"prompt_tokens": 0, "completion_tokens": 0})
        acc["prompt_tokens"] += r["uso"]["prompt_tokens"]
        acc["completion_tokens"] += r["uso"]["completion_tokens"]
    return custo_chat(por_modelo)


custo_forte = custo_chat(R_FORTE["por_modelo"])
custo_rota_chat = custo_chat(R_ROTA["por_modelo"])
custo_rota = custo_rota_chat + JEV_ROTA["cost"]
tokens_forte = sum(t["prompt_tokens"] + t["completion_tokens"] for t in R_FORTE["por_modelo"].values())
tokens_rota = sum(t["prompt_tokens"] + t["completion_tokens"] for t in R_ROTA["por_modelo"].values())
tokens_gates = JEV_ROTA["input_tokens"] + JEV_ROTA["output_tokens"]
tokens_rota_total = tokens_rota + tokens_gates
economia = 1 - custo_rota / custo_forte if custo_forte else 0.0

# Cenário: o gate rodando no jev-1.13 pago (o free pode sair do ar).
custo_gate_pago = JEV_ROTA["input_tokens"] / 1e6 * PRECO_JEV_PAGO
custo_rota_pago = custo_rota + custo_gate_pago
economia_pago = 1 - custo_rota_pago / custo_forte if custo_forte else 0.0

linhas = []
for modelo, t in R_FORTE["por_modelo"].items():
    linhas.append({"etapa": "1. all-strong (chat)", "modelo": modelo, "entrada": t["prompt_tokens"], "saída": t["completion_tokens"], "estimativa US$": custo_chat({modelo: t})})
for modelo, t in R_ROTA["por_modelo"].items():
    linhas.append({"etapa": "2. routed (chat)", "modelo": modelo, "entrada": t["prompt_tokens"], "saída": t["completion_tokens"], "estimativa US$": custo_chat({modelo: t})})
linhas.append({"etapa": "2b. gates de rota (Jev)", "modelo": os.environ.get("JEV_MODEL") or jev._DEFAULT_MODEL, "entrada": JEV_ROTA["input_tokens"], "saída": JEV_ROTA["output_tokens"], "estimativa US$": JEV_ROTA["cost"]})
linhas.append({"etapa": "2c. juiz de concordância (Jev)", "modelo": os.environ.get("JEV_MODEL") or jev._DEFAULT_MODEL, "entrada": CONCORDA["uso"]["input_tokens"], "saída": CONCORDA["uso"]["output_tokens"], "estimativa US$": CONCORDA["uso"]["cost"]})
linhas.append({"etapa": "TOTAL all-strong", "modelo": "", "entrada": sum(t["prompt_tokens"] for t in R_FORTE["por_modelo"].values()), "saída": sum(t["completion_tokens"] for t in R_FORTE["por_modelo"].values()), "estimativa US$": custo_forte})
linhas.append({"etapa": "TOTAL routed (+gates)", "modelo": "", "entrada": sum(t["prompt_tokens"] for t in R_ROTA["por_modelo"].values()) + JEV_ROTA["input_tokens"], "saída": sum(t["completion_tokens"] for t in R_ROTA["por_modelo"].values()) + JEV_ROTA["output_tokens"], "estimativa US$": custo_rota})

print(f"Economia estimada (gate free): {economia:.0%}")
print(f"Tokens (só chat): {tokens_forte} all-strong vs {tokens_rota} routed ({1 - tokens_rota / tokens_forte:.0%} menos)")
print(f"Tokens (com gates): {tokens_forte} all-strong vs {tokens_rota_total} routed ({tokens_rota_total / tokens_forte - 1:+.0%})")
print(f"Cenário gate pago (jev-1.13, US$0.042/M input): gate US$ {custo_gate_pago:.6f} -> routed US$ {custo_rota_pago:.6f} | economia {economia_pago:.0%}")
print(f"Juiz de concordância: US$ {CONCORDA['uso']['cost']:.6f} (medição - não entra na economia)")
pd.DataFrame(linhas)


Economia estimada (gate free): 79%
Tokens (só chat): 3783 all-strong vs 1504 routed (60% menos)
Tokens (com gates): 3783 all-strong vs 4254 routed (+12%)
Cenário gate pago (jev-1.13, US$0.042/M input): gate US$ 0.000107 -> routed US$ 0.000916 | economia 77%
Juiz de concordância: US$ 0.000000 (medição - não entra na economia)


,etapa,modelo,entrada,saída,estimativa US$
0,1. all-strong (chat),deepseek-v4.1-flash,699,3084,0.003910
1,2. routed (chat),glm-5.3-flash,536,554,0.000357
2,2. routed (chat),deepseek-v4.1-flash,51,363,0.000451
3,2b. gates de rota (Jev),jev-1.13-free,2558,192,0.000000
4,2c. juiz de concordância (Jev),jev-1.13-free,3154,132,0.000000
5,TOTAL all-strong,,699,3084,0.003910
6,TOTAL routed (+gates),,3145,1109,0.000808


## Experimente

**(a)** Afrouxar os critérios do `mini` — *"na dúvida, mini"*, agora com
resumo e comparação dentro. O roteador perde a desconfiança e manda mais
para o barato; a pergunta é quanto isso custa em qualidade. Economia e
concordância são medidas de novo, com as mesmas perguntas e o mesmo juiz.

Se a concordância não cair, o roteador estrito estava deixando dinheiro na
mesa. Se cair, o preço da economia apareceu — e com 6 prompts isso é
indício, não benchmark: um juiz que concorda com tudo não distingue nada.

**(b)** Rodar só as 3 moderadas com a rota invertida — forte obrigatório.
É o que se paga para não arriscar justamente onde o juiz já vinha escalando
pouco.

**(c)** Trocar `LLM_MINI_MODEL` no `.env` e rodar de novo: o notebook não
muda. O roteiro é o mesmo; o resultado é o do modelo novo — inclusive o
preço, que é a única constante que precisa ser atualizada à mão.


In [7]:
# (a) O mesmo roteiro, com o mini mais abrangente. Nenhum número é
# reaproveitado: rota, respostas e concordância são medidos de novo.
# Mudança de UMA linha: só o critério do mini, que agora cobre resumo e
# comparação e assume a dúvida. O critério do forte é o mesmo da rodada
# estrita - senão o experimento mede duas coisas ao mesmo tempo.
ROTA_FROUXA = jev.Choice(
    "Qual modelo deve responder esta mensagem?",
    {
        "mini": "Qualquer tarefa curta de saudação, formatação, classificação, resumo ou comparação simples: na dúvida, mini.",
        "forte": "Múltiplos fatores, sentimento ou precisão contextual: um erro aqui custa caro.",
    },
)
JEV_FROUXA_ANTES = jev.usage_total()
R_FROUXA = rodada(ROTEIRO, seletor=lambda p: decidir(p, ROTA_FROUXA))
JEV_FROUXA = delta(JEV_FROUXA_ANTES, jev.usage_total())
CONCORDA_FROUXA = concordancia(R_FORTE, R_FROUXA)

custo_frouxa = custo_chat(R_FROUXA["por_modelo"]) + JEV_FROUXA["cost"]
print(f"\nafrouxado: economia {1 - custo_frouxa / custo_forte:.0%} | concordância mini {pct(CONCORDA_FROUXA['taxa_mini'])} (geral {pct(CONCORDA_FROUXA['taxa_geral'])})")
print(f"estrito:   economia {economia:.0%} | concordância mini {pct(CONCORDA['taxa_mini'])} (geral {pct(CONCORDA['taxa_geral'])})")

# (b) Rota invertida nas 3 moderadas: o forte é obrigatório. A diferença
# de preço entre os dois modelos aparece inteira - sem gate para diluir.
MODERADAS = [item for item in ROTEIRO if item["tipo"] == "moderada"]
R_MOD_FORTE = rodada(MODERADAS)
custo_mod_roteadas = custo_respostas([r for r in R_ROTA["respostas"] if r["tipo"] == "moderada"])
custo_mod_forte = custo_respostas(R_MOD_FORTE["respostas"])
print(f"\n3 moderadas roteadas: US$ {custo_mod_roteadas:.6f}")
print(f"3 moderadas no forte: US$ {custo_mod_forte:.6f} ({custo_mod_forte / custo_mod_roteadas:.1f}x)")


[1/6] leve     | rota=mini conf=1.00 (mini=1.00, forte=0.00) -> glm-5.3-flash | 81 tok
          ok.


[2/6] leve     | rota=mini conf=0.98 (mini=0.99, forte=0.01) -> glm-5.3-flash | 49 tok
          Prazo apertado, o café esfria, o sono foge da janela.


[3/6] leve     | rota=mini conf=1.00 (mini=1.00, forte=0.00) -> glm-5.3-flash | 44 tok
          o prazo é o que mais importa.


[4/6] moderada | rota=mini conf=0.90 (mini=0.95, forte=0.05) -> glm-5.3-flash | 269 tok
          Essa reclamação se classifica melhor como **Problema Técnico / Bug (Falha Funcional)**, mais especificamente: 


[5/6] moderada | rota=mini conf=0.96 (mini=0.98, forte=0.02) -> glm-5.3-flash | 317 tok
          # Comparação Resposta vs. Gabarito ## O que a resposta acertou ✅ - A informação principal: **"o pedido atrasou


...

ok   moderada rota=mini  -> glm-5.3-flash        nota=0.97


ok   moderada rota=mini  -> glm-5.3-flash        nota=0.95

afrouxado: economia 87% | concordância mini 100% (geral 100%)
estrito:   economia 79% | concordância mini 100% (geral 100%)


[1/3] moderada | deepseek-v4.1-flash | 332 tok
          A melhor categoria para classificar essa reclamação é: **Problema técnico / Bug após atualização** Subcategori


[2/3] moderada | deepseek-v4.1-flash | 387 tok
          A resposta **“o pedido atrasou”** está incompleta em relação ao gabarito. Falta acrescentar: - **Por quanto te


[3/3] moderada | deepseek-v4.1-flash | 1416 tok
          - **Entrega e produto:** atrasos e rastreamento travado; pedido devolvido ao remetente sem aviso; itens com em

3 moderadas roteadas: US$ 0.000748
3 moderadas no forte: US$ 0.002053 (2.7x)
